```markdown
# Aprendizado de Máquina Automatizado (AutoML Manual): Seleção do Melhor Modelo

### Resumo
Este notebook apresenta um guia prático para comparar e selecionar automaticamente o melhor modelo de classificação para um conjunto de dados utilizando validação cruzada. Ele simula o conceito básico de AutoML, demonstrando de forma simples como testar diferentes algoritmos de Machine Learning de uma só vez para identificar qual deles apresenta o melhor desempenho.

### Objetivos de Aprendizagem
Ao final deste estudo, você será capaz de:
* Entender o conceito de pipeline de modelagem: desde a carga de dados até a avaliação final.
* Compreender a importância e o funcionamento da normalização de dados com `StandardScaler`.
* Explicar o que é validação cruzada (*Cross-Validation*) e por que ela é essencial para evitar o sobreajuste (*overfitting*).
* Criar uma estrutura simples em Python para treinar e avaliar múltiplos algoritmos concorrentemente.
* Avaliar de forma crítica os resultados obtidos no conjunto de testes com o modelo selecionado.

---

### Pré-requisitos e Bibliotecas Usadas

Para este projeto, utilizamos a biblioteca **scikit-learn** (através do import `sklearn`), que é o padrão da indústria em Python para Machine Learning clássico. Também utilizamos **pandas** e **numpy** para manipulação e análise dos dados.

* **`numpy`**: Fornece suporte para arrays multidimensionais e operações matemáticas de alta performance.
* **`pandas`**: Permite estruturar e manipular os dados em tabelas amigáveis chamadas DataFrames.
* **`StandardScaler`**: Essencial para normalizar nossos dados (deixar todos os recursos na mesma escala), o que ajuda algoritmos como KNN, SVM e Regressão Logística a funcionarem corretamente.
* **`train_test_split`**: Divide nosso conjunto de dados em uma parte para ensinar o modelo (treino) e outra para testar se ele realmente aprendeu (teste).
* **`cross_val_score`**: Executa a validação cruzada, nossa principal ferramenta para medir a qualidade geral do modelo de forma robusta.
* **`accuracy_score`**: Métrica de avaliação simples que calcula a proporção de previsões corretas sobre o total de casos.
* **Modelos de Classificação**: Importamos uma variedade de algoritmos de classificação (desde Regressão Logística a métodos baseados em árvores e redes neurais/vizinhos próximos) para criar nossa competição de modelos.
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
# cross validation score seleciona o modelo baseado em validação cruzada
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
# modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

```markdown
## Passo 1: Carregamento e Exploração Inicial dos Dados

**O que este passo faz:**
Ele carrega o conjunto de dados clássico *Breast Cancer Wisconsin* diretamente da biblioteca scikit-learn, organiza esses dados em uma tabela explicativa (DataFrame) utilizando o pandas e exibe as primeiras 5 linhas para visualização.

**Por que ele é necessário?**
Antes de qualquer modelagem, precisamos entender a estrutura dos dados. Esse dataset contém características numéricas extraídas de imagens digitalizadas de exames de mama. Nosso objetivo é classificar se o tumor é maligno ou benigno (representado pela coluna `target`).

**Como interpretar o resultado/saída:**
A saída mostra 30 colunas (características como `mean radius`, `mean texture`, etc.) para as primeiras 5 instâncias. Os valores são puramente numéricos e variam bastante de escala — por exemplo, alguns campos de área passam de 1000, enquanto valores de suavidade (*smoothness*) estão abaixo de 0.2. Essa grande diferença de escala é o motivo pelo qual precisaremos do passo de normalização mais adiante.
```

In [ ]:
# Carregar o conjunto de dados Breast Cancer
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Converter os dados em um DataFrame do pandas
df_X = pd.DataFrame(X, columns=cancer.feature_names)
df_y = pd.Series(y, name='target')

df_X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


```markdown
## Passo 2: Divisão de Dados e Normalização

**O que este passo faz:**
1. Divide as variáveis explicativas (`X`) e a variável alvo (`y`) em dois subconjuntos distintos: treino (80% dos dados) e teste (20%).
2. Aplica o `StandardScaler` para ajustar a escala dos dados de treino e aplica essa mesma transformação nos dados de teste.

**Por que ele é necessário?**
* **Divisão de dados:** Se testarmos nosso modelo nos mesmos dados que ele usou para aprender, não saberemos se ele realmente aprendeu a generalizar ou apenas decorou as respostas. A divisão simula dados do mundo real que o modelo nunca viu.
* **Normalização (`StandardScaler`):** Muitos algoritmos calculam a distância entre pontos (como KNN) ou usam gradientes para encontrar o melhor ponto (como Regressão Logística). Se uma coluna varia de 0 a 1000 e outra de 0 a 1, a coluna maior dominará o modelo injustamente. O `StandardScaler` reescala cada característica para que ela tenha média 0 e desvio padrão 1.

💡 **Para pensar:** Repare que usamos `.fit_transform()` nos dados de treino, mas apenas `.transform()` nos dados de teste. Por que isso é importante? Se usássemos o `.fit()` no teste, estaríamos deixando informações do conjunto de teste "vazarem" para a escala, o que mascararia a avaliação real do modelo (fenômeno conhecido como *data leakage*).
```

In [ ]:
# Dividir os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar os dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

```markdown
## Passo 3: Definição da Arena de Modelos

**O que este passo faz:**
Cria um dicionário em Python chamado `models` onde cada chave é o nome de um classificador e o valor é a instância da classe do respectivo modelo configurada com seus parâmetros iniciais padrões.

**Por que ele é necessário?**
Para automatizar nosso processo de seleção (AutoML Manual), precisamos organizar os candidatos. Colocá-los em um dicionário nos permite percorrer todos eles de forma sistemática usando um laço de repetição (`for`), poupando digitação e evitando repetição de código.

💡 **Para pensar:** Embora estejamos usando as configurações padrão (*default hyperparameters*) dos modelos, no mundo real, o desempenho de cada um pode mudar drasticamente se ajustarmos seus hiperparâmetros (processo conhecido como *Tuning*).
```

In [ ]:
# Definir modelos a serem avaliados
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', max_iter=10000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'SVM': SVC(gamma='scale'),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'AdaBoost': AdaBoostClassifier()
}


```markdown
## Passo 4: Competição dos Modelos via Validação Cruzada

**O que este passo faz:**
O código percorre o dicionário de modelos. Para cada um deles, realiza a **Validação Cruzada de 5 dobras** (`cv=5`) sobre os dados de treino normalizados, calcula o desempenho médio e salva o resultado.

**Por que ele é necessário?**
Se dividíssemos os dados de treino apenas uma vez para validar, poderíamos ter a sorte (ou azar) de pegar uma divisão fácil ou difícil. A **Validação Cruzada (Cross-Validation)** mitiga isso dividindo o treino em 5 partes (*folds*). O algoritmo treina em 4 partes e valida na 5ª restante. Esse processo se repete 5 vezes, garantindo que todas as partes sejam testadas. O score médio final representa uma estimativa muito mais confiável da performance real do modelo.
```

In [ ]:
# Treinar e avaliar modelos com dados normalizados
results = {}
for name, model in models.items(): # cv é o numero de fols que vai usar na validação cruzada
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    results[name] = scores.mean()

```markdown
## Passo 5: Avaliação Comparativa dos Resultados

**O que este passo faz:**
Imprime na tela a acurácia média estimada de cada modelo testado na validação cruzada.

**Por que ele é necessário?**
Permite que nós, engenheiros ou cientistas de dados, visualizemos o placar da competição para entender o comportamento de diferentes famílias de algoritmos no nosso problema específico.

**Como interpretar o resultado/saída:**
Os resultados variam de aproximadamente 91% a 98% de acurácia. Modelos baseados em comitês de decisão (como *AdaBoost*) e modelos lineares/geométricos robustos (como *Logistic Regression* e *SVM*) apresentam as melhores pontuações neste conjunto de dados, o que indica que as classes são bem separáveis linearmente ou que as regras de associação criadas pelos algoritmos de árvore se adequaram muito bem.
```

In [ ]:
# Imprimir resultados
for name, score in results.items():
    print(f'{name}: {score:.4f}')

Logistic Regression: 0.9758
Decision Tree: 0.9121
Random Forest: 0.9538
SVM: 0.9758
KNN: 0.9604
Naive Bayes: 0.9341
Gradient Boosting: 0.9516
AdaBoost: 0.9802


```markdown
## Passo 6: Seleção Automatizada do Campeão

**O que este passo faz:**
Identifica qual modelo obteve a maior nota média no dicionário de resultados e armazena a instância desse algoritmo na variável `best_model`.

**Por que ele é necessário?**
Essa é a essência do AutoML: o próprio código toma a decisão lógica de qual modelo prosseguirá para a etapa final de produção, eliminando a necessidade de alteração manual do código caso as características do dataset mudem no futuro.
```

In [ ]:
# Encontrar o melhor modelo
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

```markdown
## Passo 7: Treinamento Final do Campeão

**O que este passo faz:**
Treina (*ajusta*) o modelo vencedor utilizando **toda** a base de dados de treinamento normalizada.

**Por que ele é necessário?**
Durante a validação cruzada, o modelo foi treinado apenas em subpartes dos dados para avaliação. Agora que sabemos qual é o melhor algoritmo, queremos que ele aprenda com a maior quantidade de dados de treino disponível antes de fazermos as previsões finais de teste.
```

In [ ]:
# Treinar o melhor modelo com todos os dados de treino
# Treinar o melhor modelo com todos os dados de treino normalizados
best_model.fit(X_train_scaled, y_train)

AdaBoostClassifier()

```markdown
## Passo 8: Geração de Previsões no Teste

**O que este passo faz:**
Usa o modelo final treinado para realizar previsões de classificação para os dados de teste que foram guardados e normalizados no início.

**Por que ele é necessário?**
Neste ponto, simulamos o modelo rodando em produção. Nós fornecemos apenas os exames (`X_test_scaled`) sem contar ao modelo quais são as respostas corretas, testando sua capacidade real de dedução.
```

In [ ]:
# Fazer previsões no conjunto de teste normalizado
y_pred = best_model.predict(X_test_scaled)

```markdown
## Passo 9: Avaliação Final no Conjunto de Teste

**O que este passo faz:**
Compara as respostas que o modelo gerou (`y_pred`) com os gabaritos reais guardados (`y_test`) e calcula a acurácia final.

**Por que ele é necessário?**
Este é o veredito final do nosso pipeline de aprendizado de máquina. Ele nos dá a métrica de segurança de quão bem nosso modelo se comportará diante de novos pacientes reais.

**Como interpretar o resultado/saída:**
A saída exibe uma acurácia alta (geralmente acima de 95%). Isso significa que nosso pipeline automatizado foi extremamente bem-sucedido em identificar um modelo capaz de prever corretamente a presença ou ausência de câncer de mama na grande maioria dos casos de teste.
```

In [ ]:
# Avaliar o desempenho no conjunto de teste
accuracy = accuracy_score(y_test, y_pred)
print(f'Melhor modelo ({best_model_name}) - Acurácia no teste: {accuracy:.4f}')

Melhor modelo (AdaBoost) - Acurácia no teste: 0.9649


```markdown
---

## Erros Comuns e Dicas de Sobrevivência

1. **Vazamento de Dados (*Data Leakage*):** Um erro clássico de iniciantes é aplicar o `.fit_transform()` do `StandardScaler` em todo o dataset antes da divisão de treino e teste. Se fizer isso, a média e o desvio padrão dos dados de teste influenciarão os dados de treino, gerando resultados otimistas ilusórios no treino que falharão miseravelmente em produção.
2. **Uso incorreto de dados escalados:** Lembre-se sempre de passar `X_train_scaled` e `X_test_scaled` para os modelos. Se você acidentalmente passar o `X_train` sem normalização, alguns algoritmos (como SVM e KNN) apresentarão desempenhos severamente reduzidos.
3. **Cuidado com Desbalanceamento de Classes:** O dataset de câncer de mama é relativamente equilibrado. Porém, em problemas reais onde uma classe é muito rara (como fraudes bancárias de 0.1%), usar apenas **Acurácia** pode ser enganoso. Um modelo que chuta sempre que "não é fraude" terá 99.9% de acurácia, mas será totalmente inútil.

---

## Resumo do que Aprendemos

* **AutoML na prática:** Criar um dicionário de modelos e avaliá-los em lote permite economizar tempo e encontrar o melhor ponto de partida rapidamente.
* **Normalização:** Manter as escalas das features uniformes é vital para modelos geométricos e baseados em distância.
* **Validação Cruzada:** É a ferramenta mais robusta para estimar o desempenho real de um modelo em treino sem sofrer com a flutuação estatística de uma divisão simples.

---

## Exercícios Propostos (Sem Gabarito)

1. **[Fácil] Mudança de Métrica:** Altere o código de avaliação da validação cruzada para reportar e selecionar o melhor modelo com base na métrica de **Precisão** (*precision*) ou **Revocação** (*recall*) ao invés da acurácia padrão do `cross_val_score`.
2. **[Médio] Impacto da Escala:** Remova a etapa de normalização (`StandardScaler`) e rode o pipeline diretamente sobre `X_train` e `X_test`. Quais modelos mantiveram o desempenho e quais falharam drasticamente? Justifique o motivo com base no funcionamento interno dos algoritmos.
3. **[Desafiador] Adicionando Hiperparâmetros ao AutoML:** Modifique a estrutura de dicionário de modelos para que ela não teste apenas modelos diferentes, mas também variações de parâmetros (ex: testar KNN com `n_neighbors=3`, `n_neighbors=5` e `n_neighbors=11`) e descubra qual combinação é a vencedora absoluta.

---

## Para ir Além

Para expandir seus conhecimentos em automação e engenharia de machine learning, pesquise sobre os seguintes termos:
* **GridSearchCV e RandomizedSearchCV:** Ferramentas prontas do Scikit-Learn que automatizam a busca pelos melhores hiperparâmetros de um modelo.
* **Bibliotecas de AutoML de mercado:** Pesquise sobre ferramentas profissionais como o *TPOT*, *Auto-Sklearn* e *H2O AutoML*, que realizam buscas complexas por algoritmos e tratamentos de dados de forma 100% automatizada.
* **Métricas de Matriz de Confusão, F1-Score e Curva ROC-AUC:** Fundamentais para avaliar modelos de classificação em cenários mais desafiadores e com dados desbalanceados.
```